In [37]:
import os
import ollama
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

True

In [38]:
# Available ollama models locally


def _format_size(size_bytes: int) -> str:
    size = float(size_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024


OllamaHost = os.getenv("OLLAMA_HOST")
OllamaClient = ollama.Client(host=OllamaHost)
OllamaClient.generate
available_models = [
    {
        "model": m.model,
        "size": _format_size(m.size or 0),
        "params": m.details.parameter_size if m.details else "—",
        "quant": m.details.quantization_level if m.details else "—",
        "modified": m.modified_at.strftime("%Y-%m-%d %H:%M") if m.modified_at else "—",
    }
    for m in OllamaClient.list().models
]

rows = "\n".join(f"{m['model']}" for m in available_models)
print(rows)

qwen3.6:35b
gemma4:31b
gpt-oss:20b
gemma4:26b
nomic-embed-text:latest
gemma4:e4b


In [39]:
MODEL = "gemma4:e4b"

In [ ]:
system_prompt = f"""
You are a synthetic data generator called DataAlchemy.
You will be given a topic or idea and based on that you will generate a realistic dataset.
You will need these 3 things from the user, if don't have them, ask user to provide them (user can also choose to skip):
1. Topic or idea.
2. Complexity of the dataset: 
    - Simple: not much variation with less linking between entities and less data fields; less rows
    - Medium: some variation with more linking between entities and more data fields; more rows
    - Hard: lots of variation with lots of linking between entities and lots of data fields; lots of rows
3. Format of the dataset: supported formats are CSV, JSON and Markdown.

You will generate a dataset based on the above requirements and return it in the requested format.
In the end should give a zip file with the dataset and a description of the dataset.

When you are asking for more info reply like below:
{
    "type": "missing_topic",
    "message": "I need a topic or idea to generate the dataset. Please provide a topic or idea."
}
{
    "type": "missing_complexity",
    "message": "I will need complexity level of the dataset. Please select one of the following: Simple, Medium, Hard"
}
{
    "type": "missing_format",
    "message": "I will need format of the dataset. Please select one of the following: CSV, JSON, Markdown"
}

When you are done with the dataset, reply like below:
{
    "type": "done",
    "message": "I have generated the dataset. Please find the zip file with the dataset and a description of the dataset."
}
"""

In [ ]:
openai = OpenAI(base_url=OllamaHost, api_key="ollama")

topic = ""
complexity = ""
format = ""
chat_history = []


def generate_dataset(user_prompt, history):
    chat_history = history if history else []
    chat_history.append({"role": "user", "content": user_prompt})
    result = openai.chat.completions.create(
        model=MODEL,
        messages=chat_history,
    )
    generated_text = result.choices[0].message.content
    chat_history.append({"role": "assistant", "content": generated_text})
    return generated_text